## **🫱고령 남성의 손 구조 유형 분류 – 손가락 마디 및 손둘레 기반 군집 분석**

### **📜분석절차**
1. **데이터전처리 - 고령 남성 추출, 변수선택, 결측치처리**
2. **T-test(2030 vs 70↑) : 연령이 손 구조에 영향을 주는가?**
4. **정규화 - 변수 표준화 z-score (평균0 표준편차1)**
5. **요인분석 - 비슷한 변수끼리 묶음**  
6. **군집분석 - 손 구조 유형 분류**
7. **군집해석**
8. **결론**

🔎**분석항목**

- 너비
    - 검지손가락끝마디너비 / 가운데손가락끝마디너비 / 반지손가락끝마디너비 / 새끼손가락끝마디너비 /
검지손가락중간마디너비 / 가운데손가락중간마디너비 / 반지손가락중간마디너비 / 새끼손가락중간마디너비

- 둘레
    - 검지손가락끝마디둘레 / 가운데손가락끝마디둘레 / 반지손가락끝마디둘레 / 새끼손가락끝마디둘레 /
검지손가락중간마디둘레 / 가운데손가락중간마디둘레 / 반지손가락중간마디둘레 / 새끼손가락중간마디둘레 / 손둘레

- 길이
    - 손직선길이 / 검지손가락직선길이 / 가운데손가락직선길이 / 반지손가락직선길이 / 새끼손가락직선길이 /
손바닥직선길이 / 손안쪽가쪽직선길이

- 두께
    - 손두께 / 손두께(2/3 지점)


### **1. 데이터전처리 - 고령 남성 추출, 변수선택, 결측치처리**

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

#70세 이상 손 치수데이터
df = pd.read_csv('C:/Users/FORYOUCOM/Desktop/인체치수조사_치수데이터(공개용)/고령손_8차 인체치수조사(2020~23).csv', 
                 encoding='utf-8')

#남성만 필터링
df= df.loc[df['성별']=='남', :]

#결측치 확인결과 -> 결측치 없음
# df.isnull().sum()

#앞에 붙은 숫자 제거  (336. 손직선길이 → 손직선길이)
old_cols = df.columns[3:]
new_cols = []
for col in old_cols:
    temp = col.split('.')[1].strip()
    new_cols.append(temp)
df.rename(columns=dict(zip(old_cols, new_cols)), inplace=True)

#분석항목 필터링
df.drop([
    # 첫마디 관련 길이 - 중간/끝마디와 유사 패턴, 정보 중복
    '손목중심-엄지손가락첫마디길이',
    '손목중심-검지손가락첫마디길이',
    '손목중심-반지손가락첫마디길이',
    '손목중심-새끼손가락첫마디길이',

    # 복합구간 길이 - 측정 지점이 복잡하고 해석 어려움
    '손목가쪽-엄지손가락손끝길이',
    '엄지손가락첫마디-검지손가락첫마디직선길이',

    # 첫마디 둘레 - 중복 정보 많고 영향 낮음으로 판단
    '엄지손가락첫마디둘레',
    '검지손가락첫마디둘레',
    '가운데손가락첫마디둘레',
    '반지손가락첫마디둘레',
    '새끼손가락첫마디둘레'
], axis=1, inplace=True)

In [208]:
df.shape

(211, 31)

In [32]:
df.describe()

,나이,손직선길이,엄지손가락직선길이,검지손가락직선길이,가운데손가락직선길이,반지손가락직선길이,새끼손가락직선길이,손바닥직선길이,검지손가락끝마디너비,가운데손가락끝마디너비,...,가운데손가락끝마디둘레,반지손가락끝마디둘레,새끼손가락끝마디둘레,검지손가락중간마디둘레,가운데손가락중간마디둘레,반지손가락중간마디둘레,새끼손가락중간마디둘레,손둘레,손두께,손두께(2/3 지점)
count,211.000000,211.000000,211.000000,211.000000,211.000000,211.000000,211.000000,211.000000,211.000000,211.000000,...,211.000000,211.000000,211.000000,211.000000,211.000000,211.000000,211.000000,211.000000,211.000000,211.000000
mean,75.867299,181.435071,52.624645,70.437915,78.070616,74.616588,58.906635,103.425118,18.462085,18.581517,...,58.667773,55.452133,51.584360,67.752607,69.555450,66.786256,59.107583,215.490995,31.856872,44.056872
std,3.844777,7.824388,3.962450,4.655359,5.051071,4.702336,4.999500,5.400034,1.154817,1.205973,...,3.083088,3.101925,4.306594,3.411218,3.692374,3.457187,3.960476,8.957581,2.418360,2.795113
min,70.000000,159.800000,41.100000,57.200000,61.100000,61.100000,41.100000,86.200000,15.100000,15.100000,...,51.700000,45.700000,41.400000,60.000000,59.300000,58.600000,50.000000,194.000000,26.200000,38.000000
25%,73.000000,176.650000,50.250000,67.400000,74.850000,71.200000,55.650000,100.000000,17.700000,17.800000,...,56.500000,53.400000,49.200000,65.100000,67.000000,64.200000,56.350000,209.750000,30.100000,42.100000
50%,75.000000,181.400000,53.000000,70.700000,78.900000,74.800000,59.200000,103.300000,18.500000,18.600000,...,58.400000,55.400000,51.500000,67.800000,69.400000,66.700000,59.100000,215.800000,32.000000,43.900000
75%,79.000000,186.950000,55.450000,73.700000,81.350000,77.800000,62.800000,107.000000,19.250000,19.300000,...,60.500000,57.300000,53.900000,70.150000,72.100000,68.900000,61.350000,221.600000,33.500000,45.700000
max,84.000000,203.800000,62.600000,83.300000,90.600000,87.100000,70.100000,116.500000,21.700000,22.000000,...,67.900000,64.200000,69.200000,79.500000,80.900000,76.600000,71.800000,243.300000,38.400000,53.100000


### **2. T-test(2030 vs 70↑) : 연령이 손 구조에 영향을 주는가?**

In [245]:
#20~69세 전체 치수데이터
young_df = pd.read_csv('C:/Users/FORYOUCOM/Desktop/인체치수조사_치수데이터(공개용)/3D_8차 인체치수조사(2020~23).csv', 
                 encoding='utf-8', header=6)

#기본정보와 손 컬럼만 필터링
young_df = young_df.iloc[:, np.r_[1:4, 287:326]]

#남성만 필터링
young_df= young_df.loc[young_df['성별']=='M',:]

#결측치제거
young_df=young_df.dropna()

# #2030 연령대만 필터링
young_df = young_df[(young_df['나이']>=20) & (young_df['나이']<40)]

#앞에 붙은 숫자 제거  (336. 손직선길이 → 손직선길이)
old_cols = young_df.columns[3:]
new_cols = []
for col in old_cols:
    temp = col.split('.')[1].strip()
    new_cols.append(temp)

young_df.rename(columns=dict(zip(old_cols, new_cols)), inplace=True)

#분석항목 필터링
young_df.drop([
    # 첫마디 관련 길이 - 중간/끝마디와 유사 패턴, 정보 중복
    '손목중심-엄지손가락첫마디길이',
    '손목중심-검지손가락첫마디길이',
    '손목중심-반지손가락첫마디길이',
    '손목중심-새끼손가락첫마디길이',

    # 복합구간 길이 - 측정 지점이 복잡하고 해석 어려움
    '손목가쪽-엄지손가락손끝길이',
    '엄지손가락첫마디-검지손가락첫마디직선길이',

    # 첫마디 둘레 - 중복 정보 많고 영향 낮음으로 판단
    '엄지손가락첫마디둘레',
    '검지손가락첫마디둘레',
    '가운데손가락첫마디둘레',
    '반지손가락첫마디둘레',
    '새끼손가락첫마디둘레'
], axis=1, inplace=True)

In [269]:
young_df.shape

(1342, 31)

In [268]:
import scipy.stats as stats

hand_cols = df.columns[3:]

#t-test (젊은남성 손 컬럼 vs 고령남성 손 컬럼)
for col in hand_cols:
    g1 = df.loc[:, col]
    g2 = young_df.loc[:, col]
    t_stat, p_value = stats.ttest_ind(g1, g2, equal_var=True)

    print(col)
    print(f"    F값 : {f_stat:.2f}")
    print(f"    p값 : {p_value:.2f}")

손직선길이
    F값 : 3.50
    p값 : 0.00
엄지손가락직선길이
    F값 : 3.50
    p값 : 0.00
검지손가락직선길이
    F값 : 3.50
    p값 : 0.34
가운데손가락직선길이
    F값 : 3.50
    p값 : 0.01
반지손가락직선길이
    F값 : 3.50
    p값 : 0.00
새끼손가락직선길이
    F값 : 3.50
    p값 : 0.00
손바닥직선길이
    F값 : 3.50
    p값 : 0.01
검지손가락끝마디너비
    F값 : 3.50
    p값 : 0.00
가운데손가락끝마디너비
    F값 : 3.50
    p값 : 0.00
반지손가락끝마디너비
    F값 : 3.50
    p값 : 0.00
새끼손가락끝마디너비
    F값 : 3.50
    p값 : 0.00
검지손가락중간마디너비
    F값 : 3.50
    p값 : 0.00
가운데손가락중간마디너비
    F값 : 3.50
    p값 : 0.00
반지손가락중간마디너비
    F값 : 3.50
    p값 : 0.00
새끼손가락중간마디너비
    F값 : 3.50
    p값 : 0.00
손안쪽가쪽직선길이
    F값 : 3.50
    p값 : 0.00
엄지손가락끝마디둘레
    F값 : 3.50
    p값 : 0.00
검지손가락끝마디둘레
    F값 : 3.50
    p값 : 0.00
가운데손가락끝마디둘레
    F값 : 3.50
    p값 : 0.00
반지손가락끝마디둘레
    F값 : 3.50
    p값 : 0.00
새끼손가락끝마디둘레
    F값 : 3.50
    p값 : 0.00
검지손가락중간마디둘레
    F값 : 3.50
    p값 : 0.00
가운데손가락중간마디둘레
    F값 : 3.50
    p값 : 0.00
반지손가락중간마디둘레
    F값 : 3.50
    p값 : 0.00
새끼손가락중간마디둘레
    F값 : 3.50
    p값 : 0.00
손둘레
    F값 : 3.50
    p값 :

### **3. 정규화 - 변수 표준화(평균0 표준편차1)**
손둘레(180) vs 손가락너비(15) 같이 단위 차이 매우 큼
각 변수의 단위와 크기가 다르므로 표준화 (z-score) 수행 → 평균 0, 표준편차 1로 맞춰줌

In [34]:
from sklearn.preprocessing import StandardScaler
\
s


caler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df.iloc[:,3:]), columns=df.columns[3:])
df_scaled

,손직선길이,엄지손가락직선길이,검지손가락직선길이,가운데손가락직선길이,반지손가락직선길이,새끼손가락직선길이,손바닥직선길이,검지손가락끝마디너비,가운데손가락끝마디너비,반지손가락끝마디너비,...,가운데손가락끝마디둘레,반지손가락끝마디둘레,새끼손가락끝마디둘레,검지손가락중간마디둘레,가운데손가락중간마디둘레,반지손가락중간마디둘레,새끼손가락중간마디둘레,손둘레,손두께,손두께(2/3 지점)
0,1.917149,0.676783,1.089953,1.732334,1.488626,0.259314,1.146209,-1.355885,-0.400226,0.418343,...,-0.412180,0.597133,-0.601521,-0.750077,-0.286526,-0.198973,-0.280324,0.605283,-1.059790,-0.163843
1,0.149238,-0.815736,-1.106281,-0.986412,0.294896,-0.322123,1.127646,-0.835087,-0.898933,-1.050110,...,-1.290008,-1.309435,-0.694623,-1.602235,-1.643888,-0.604890,0.251176,-0.457794,-0.603854,-1.203835
2,0.636054,2.219895,-0.826368,-0.252152,0.230947,0.299413,1.146209,-1.442685,-1.646993,-0.968529,...,-1.842714,-1.729527,-1.788568,-1.132079,-1.481005,-1.706663,-1.900134,-0.368271,-2.220352,-0.737632
3,2.865159,1.688660,2.188070,2.486439,2.661039,1.803131,1.814457,1.595305,0.347834,0.907827,...,0.335599,0.338615,-0.112737,0.924854,1.070837,0.525878,-0.330943,0.952181,-1.764417,-0.056257
4,0.405457,0.196141,0.099495,0.025676,-0.003536,-0.101578,0.570773,-1.095486,-0.233990,-1.131690,...,-0.477204,-1.050917,-1.579089,-0.897001,-0.693735,-0.981812,-1.191467,-0.636838,-0.686752,-1.132112
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,-0.158224,1.283909,-1.278534,-0.529980,-0.344602,-1.304552,0.255212,0.466908,0.763423,-3.987014,...,-0.119571,-1.826471,0.585526,-0.838231,-0.286526,-1.213765,-0.027229,-0.413033,0.308017,-1.060388
207,-0.580985,0.626189,-0.611051,-0.668894,-1.303849,-2.407278,-0.227412,-1.008687,-0.566462,-0.968529,...,-0.867350,-0.533882,-2.323902,-1.249618,-0.693735,-0.981812,-1.368634,-2.002052,-1.598622,-0.916941
208,-0.606607,0.575595,-0.675647,-0.609360,-1.175949,-2.347130,-0.338787,-1.095486,-0.483344,-0.886948,...,-0.834838,-0.501567,-2.370453,-1.279003,-0.666587,-0.981812,-1.545801,-1.968481,-1.681520,-0.881079
209,0.046751,-0.259204,-0.331139,0.243970,-0.685667,-1.224354,-0.171725,0.119710,0.015363,0.173601,...,0.140526,0.241671,0.259670,-0.044843,0.663628,0.380907,-0.584039,-0.547316,-0.728200,0.338223


### **4. 요인분석 - 비슷한 변수끼리 묶음**

In [43]:
from factor_analyzer import FactorAnalyzer
import matplotlib.pyplot as plt
import seaborn as sns

fa = FactorAnalyzer(n_factors=2, rotation='varimax')
fa.fit(df.iloc[:,3:])

loadings = pd.DataFrame(fa.loadings_, index=df.columns[3:], columns=["요인1", "요인2"])
loadings.round(2)

,요인1,요인2
손직선길이,0.31,0.70
엄지손가락직선길이,0.06,0.58
검지손가락직선길이,0.12,0.80
가운데손가락직선길이,0.19,0.89
반지손가락직선길이,0.15,0.89
새끼손가락직선길이,0.03,0.70
손바닥직선길이,0.26,0.21
검지손가락끝마디너비,0.77,0.01
가운데손가락끝마디너비,0.78,0.12
반지손가락끝마디너비,0.75,0.13


In [44]:
fa = FactorAnalyzer(n_factors=6, rotation='varimax')
fa.fit(df.iloc[:,3:])

loadings = pd.DataFrame(fa.loadings_, index=df.columns[3:], columns=["요인1", "요인2","요인3","요인4", "요인5","요인6"])
loadings.round(2)

,요인1,요인2,요인3,요인4,요인5,요인6
손직선길이,0.27,0.65,0.03,0.07,0.70,0.03
엄지손가락직선길이,0.06,0.57,0.05,0.00,0.11,-0.06
검지손가락직선길이,0.08,0.82,0.01,0.23,-0.00,0.08
가운데손가락직선길이,0.27,0.89,-0.01,-0.00,0.06,0.06
반지손가락직선길이,0.14,0.89,0.13,0.01,0.05,-0.04
새끼손가락직선길이,-0.06,0.71,0.23,0.00,-0.02,0.05
손바닥직선길이,0.15,0.09,0.07,0.10,0.86,-0.03
검지손가락끝마디너비,0.58,0.00,0.24,0.50,0.16,-0.28
가운데손가락끝마디너비,0.78,0.10,0.26,0.06,0.11,-0.13
반지손가락끝마디너비,0.64,0.10,0.36,0.16,0.16,-0.13


### **5. 군집분석 - 손 구조 유형 분류**

### **6. 군집해석**

### **7. 결론**